# Exploratory Data Analysis

## Import packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [2]:
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt

### Data Overview

In [3]:
con = duckdb.connect("lol_predictors.duckdb")

Feature selection

In [ ]:
def data_selection(df, included_cols):
    # Choose the dataset using a SQL query
    df = con.execute(f"""
    SELECT {", ".join(included_cols)}
    FROM {df}
    WHERE position = 'team';
    """).df()

    return df

In [ ]:
# Adjust list based on the features considered
cols_to_include = ['gameid', 'league', 'year', 'split', 'patch',
                   'teamname', 'gamelength', 'result', 'kills',
                   'deaths', 'assists', 'towers', 'dpm', 'damagetakenperminute',
                   'wardskilled', 'totalgold', 'goldspent'
                   ]

In [ ]:
df_selected = "df_2014" # temporarily hardcoded df name
df = data_selection(df_selected, cols_to_include)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1836 entries, 0 to 1835
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   gameid                1836 non-null   str    
 1   league                1836 non-null   str    
 2   year                  1836 non-null   int64  
 3   split                 1680 non-null   str    
 4   patch                 1478 non-null   float64
 5   teamname              1836 non-null   str    
 6   gamelength            1836 non-null   int64  
 7   result                1836 non-null   int64  
 8   kills                 1836 non-null   int64  
 9   deaths                1836 non-null   int64  
 10  assists               1836 non-null   int64  
 11  towers                1836 non-null   float64
 12  dpm                   1836 non-null   float64
 13  damagetakenperminute  1836 non-null   float64
 14  wardskilled           1836 non-null   int64  
 15  totalgold             1836 non-n

In [13]:
df.shape

(1836, 17)

In [15]:
df.describe()

,year,patch,gamelength,result,kills,deaths,assists,towers,dpm,damagetakenperminute,wardskilled,totalgold,goldspent
count,1836.000000,1478.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000,1836.000000
mean,2014.113290,4.091083,2281.938998,0.500000,13.464052,13.486928,31.408497,6.459150,1897.535525,2688.462071,27.109477,61644.996187,54810.416667
std,0.317033,0.129632,518.344662,0.500136,7.458098,7.458904,19.057887,3.412266,467.734221,460.097915,13.230487,15686.132262,15237.440733
min,2014.000000,3.150000,1218.000000,0.000000,0.000000,0.000000,0.000000,0.000000,595.044900,1323.739500,1.000000,23991.000000,20583.000000
25%,2014.000000,4.070000,1920.000000,0.000000,7.000000,8.000000,16.000000,3.000000,1569.569325,2366.829975,17.000000,50453.750000,43937.000000
50%,2014.000000,4.110000,2220.500000,0.500000,13.000000,13.000000,30.000000,7.000000,1880.978450,2675.609300,25.000000,61117.000000,53084.000000
75%,2014.000000,4.140000,2569.000000,1.000000,18.000000,18.000000,45.000000,10.000000,2200.015275,2979.522800,35.000000,71823.750000,63962.750000
max,2015.000000,4.210000,4837.000000,1.000000,51.000000,51.000000,142.000000,11.000000,3802.547300,4663.127400,87.000000,129459.000000,127251.000000


# Data Audit

In [ ]:
def data_profiling(df, df_selected=df_selected):
    """
    Present the profile of the dataframe, including
    description, number of missing values, number of unique values, and unique values of object series
    """    
    with pd.option_context('display.max_seq_items', None): # list all columns of the df with no limits
        print(f"List of Columns: {df.columns}")
    
    with pd.option_context('display.max_rows', None): # list all rows of the df with no limits
        # State the number of missing items for each column
        print(f"\n{" COUNT OF MISSING VALUES ":-^50}") 
        display(df.isnull().sum())

        print(f"\n{" COUNT OF UNIQUE VALUES ":-^50}")
        display(df.nunique())

        print(f"\n{" UNIQUE VALUES OF OBJECT SERIES ":-^50}")
        for object_column in df.select_dtypes(include=['object']):
            if object_column == 'gameid': # skip gameid series
                continue
            print("\n",df[object_column].value_counts())


In [ ]:
def count_outliers(df, IQR_multiplier=1.5):
    """
    Utilizes IQR to detect outliers.
    """
    # Define numerical columns
    num_cols = df.select_dtypes(include=["number"]).columns.drop(['year', 'patch', 'result']).tolist()

    print(f"\n{" COUNT OF OUTLIERS ":-^50}")
    for col in num_cols:
        # Setting the IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        # Standard 1.5 multiplier for outlier bounds
        lower_bound = Q1 - IQR_multiplier * IQR
        upper_bound = Q3 + IQR_multiplier * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        print(f"{f"{col}:":<25} {len(outliers):>5}")

# Data Visualisation

In [ ]:
def visualize_outliers(df):
    """
    Utilize box plots to detect outliers.
    """
    # Define numerical columns to visualize
    num_cols = df.select_dtypes(include=["number"]).columns.drop(['year', 'patch', 'result']).tolist()
    fig, axes = plt.subplots(3, 4, figsize=(16, 12)) # Adjust depending on the number of numerical columns
    axes = axes.flatten()

    for i, col in enumerate(num_cols):
        sns.boxplot(y=df[col], ax=axes[i])
        axes[i].set_title(col, fontsize=12, fontweight="bold")
        axes[i].set_ylabel("")

    # Hide unused subplots
    for j in range(len(num_cols), len(axes)):
        fig.delaxes(axes[j])

    print(f"{" BOXPLOTS OF NUMERICAL COLUMNS ":-^50}")
    plt.tight_layout()
    plt.show()

In [2]:
def visualize_distribution():
    pass

In [4]:
def visualize_correlations():
    pass

In [4]:
con.close()